# Multimodal Financial Market Analysis — BBCA
### NeuralProphet: Price-Only Baseline vs Hybrid (Price + Fundamentals)

**Authors**: Wesley Coa, Geoffrey Gohtama, Wilbert — Bina Nusantara University, 2026

**Fixes applied:**  
1. `set_plotting_backend("matplotlib")` — prevents `suptitle` AttributeError  
2. Forecast aligned to actuals via `ds` date-merge (not fragile `iloc`)  
3. `n_lags` raised 1 → 14 for stronger autoregressive signal  
4. Proper out-of-sample evaluation (train on train slice only)  


## Imports, Configuration & Backend Setup

In [25]:
"""
================================================================================
  Multimodal Financial Market Analysis — BBCA Stock
  Fusing Price Time-Series + Fundamental Financial Report Features
  via NeuralProphet External Regressors
================================================================================
  Authors : Wesley Coa, Geoffrey Gohtama, Wilbert
  Thesis  : "Multimodal Financial Market Analysis: Fusing Price Time-Series
             and Corporate Report Text Features"
  School  : Universitas Bina Nusantara, 2026
================================================================================

  FIXES applied in this version:
    1. Force matplotlib plotting backend to avoid Plotly Figure.suptitle() error
    2. Align forecasts to actuals by `ds` date column (not fragile iloc positions)
    3. n_lags 1→7 (good AR signal without 4-hour training runs)
    4. Proper out-of-sample test: model is fit on train portion only
    5. plot_components() REPLACED with manual matplotlib decomposition plot —
       NeuralProphet's built-in call hangs when plotly-resampler is absent;
       reading component columns directly is faster and gives identical info
================================================================================
"""

# ── Standard library ──────────────────────────────────────────────────────────
import json
import warnings
warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — change to "TkAgg" if
import matplotlib.pyplot as plt  # you want interactive pop-up windows
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── NeuralProphet — MUST force matplotlib BEFORE any NP import ────────────────
from neuralprophet import NeuralProphet, set_log_level
set_log_level("ERROR")         # suppress verbose training output

# Force matplotlib so plot_components() returns a matplotlib Figure (not Plotly)
try:
    from neuralprophet import set_plotting_backend
    set_plotting_backend("matplotlib")
except ImportError:
    pass  # older NeuralProphet versions don't need this call


# ==============================================================================
#  CONFIGURATION  — edit these paths to match your local folder layout
# ==============================================================================

PRICE_DATA_PATH  = "BBCA.JK_dataprice.xlsx"
FUNDAMENTAL_PATH = "BBCA_fundamental.json"

TEST_SIZE        = 120          # last N trading days = held-out test set
EPOCHS           = 50           # 50 is sufficient; 150 caused 4-hour runs
LEARNING_RATE    = 0.005        # slightly lower LR for stability
N_LAGS           = 7            # AR window: 7 trading days (1 week); 14 was too slow
N_FORECASTS      = 1            # 1-step-ahead prediction

# Fundamental feature columns used as lagged external regressors
REGRESSOR_COLS = [
    "net_income",
    "revenue",
    "net_profit_margin",
    "ni_growth_qoq",
    "rev_growth_qoq",
]

## Section 1 — Load Price Data

In [26]:
# ==============================================================================
#  SECTION 1 — LOAD PRICE DATA
# ==============================================================================

def load_price_data(path: str) -> pd.DataFrame:
    df_raw = pd.read_excel(path)
    df_raw["Date"] = pd.to_datetime(df_raw["Date"])
    df_raw = df_raw.sort_values("Date").reset_index(drop=True)
    df = df_raw.rename(columns={"Date": "ds", "Close": "y"})[["ds", "y"]]
    print(f"[Price Data]  {df['ds'].min().date()} → {df['ds'].max().date()} "
          f"({len(df)} trading days)")
    return df

## Section 2 — Load & Parse Fundamental JSON

In [27]:
# ==============================================================================
#  SECTION 2 — LOAD & PARSE FUNDAMENTAL JSON
# ==============================================================================

_QUARTER_END = {"Q1": (3, 31), "Q2": (6, 30), "Q3": (9, 30), "Q4": (12, 31)}


def _parse_value(raw: str) -> float:
    return float(raw.replace(" B", "").replace(",", ""))


def parse_quarterly_series(financial_year_values: list, col_name: str) -> pd.DataFrame:
    records = []
    for year_block in financial_year_values:
        year = int(year_block["year"])
        for period in year_block["period_values"]:
            q = period["period"]
            month, day = _QUARTER_END[q]
            records.append({
                "ds":     pd.Timestamp(year=year, month=month, day=day),
                col_name: _parse_value(period["quarter_value"]),
            })
    return pd.DataFrame(records).sort_values("ds").reset_index(drop=True)


def load_fundamental_data(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        data = json.load(f)

    groups   = data["data"]["financial_year_parent"]["financial_year_groups"]
    ni_years  = next(g["financial_year_values"] for g in groups
                     if g["fitem_name"] == "Net Income")
    rev_years = next(g["financial_year_values"] for g in groups
                     if g["fitem_name"] == "Revenue")

    ni_df  = parse_quarterly_series(ni_years,  "net_income")
    rev_df = parse_quarterly_series(rev_years, "revenue")

    fund = ni_df.merge(rev_df, on="ds")
    fund["net_profit_margin"] = fund["net_income"] / fund["revenue"]
    fund["ni_growth_qoq"]     = fund["net_income"].pct_change()
    fund["rev_growth_qoq"]    = fund["revenue"].pct_change()

    # Fill the very first NaN growth values with 0
    fund[["ni_growth_qoq", "rev_growth_qoq"]] = \
        fund[["ni_growth_qoq", "rev_growth_qoq"]].fillna(0)

    print(f"\n[Fundamental Data]  {len(fund)} quarterly observations "
          f"({fund['ds'].min().date()} → {fund['ds'].max().date()})")
    print(fund.to_string(index=False))
    return fund

## Section 3 — Align Fundamentals to Daily (Forward-Fill)

In [28]:
# ==============================================================================
#  SECTION 3 — ALIGN FUNDAMENTALS TO DAILY FREQUENCY (FORWARD-FILL)
# ==============================================================================

def align_fundamentals_to_daily(
    df_price: pd.DataFrame,
    fund_quarterly: pd.DataFrame,
) -> pd.DataFrame:
    """
    Forward-fill quarterly fundamental values to every calendar day, then
    inner-join to actual trading days.  This replicates the information set
    available to an investor: once a report is published its values persist
    until the next quarterly release.
    """
    date_range = pd.DataFrame({
        "ds": pd.date_range(df_price["ds"].min(), df_price["ds"].max(), freq="D")
    })

    daily = date_range.merge(fund_quarterly, on="ds", how="left")
    daily[REGRESSOR_COLS] = daily[REGRESSOR_COLS].ffill()
    daily = daily.dropna(subset=["net_income"]).reset_index(drop=True)

    df_hybrid = df_price.merge(daily, on="ds", how="inner").reset_index(drop=True)
    df_hybrid = df_hybrid.dropna(subset=REGRESSOR_COLS).reset_index(drop=True)

    print(f"\n[Hybrid Dataset]  {len(df_hybrid)} rows after alignment "
          f"({df_hybrid['ds'].min().date()} → {df_hybrid['ds'].max().date()})")
    return df_hybrid

## Section 4 — Normalise Regressors

In [29]:
# ==============================================================================
#  SECTION 4 — NORMALISE REGRESSORS
# ==============================================================================

def normalise_regressors(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        mu, sigma = df[col].mean(), df[col].std()
        df[col] = (df[col] - mu) / sigma if sigma > 0 else 0.0
    return df

## Section 5 — Build NeuralProphet Model

In [30]:
# ==============================================================================
#  SECTION 5 — BUILD NeuralProphet MODEL
# ==============================================================================

def build_model(with_regressors: bool = False) -> NeuralProphet:
    model = NeuralProphet(
        n_lags             = N_LAGS,
        n_forecasts        = N_FORECASTS,
        learning_rate      = LEARNING_RATE,
        seasonality_mode   = "additive",
        epochs             = EPOCHS,
        yearly_seasonality = True,
        weekly_seasonality = True,
        daily_seasonality  = False,
        quantiles          = [0.05, 0.95],
    )
    model.add_seasonality(name="quarterly", period=91.31, fourier_order=5)

    if with_regressors:
        for col in REGRESSOR_COLS:
            model.add_lagged_regressor(names=col, n_lags=N_LAGS)

    return model

## Section 6 — Train & Generate Predictions

In [31]:
# ==============================================================================
#  SECTION 6 — TRAIN & GENERATE IN-SAMPLE PREDICTIONS
#              (fit on train slice, predict over entire dataset so we can
#               extract the test-period rows by date for fair evaluation)
# ==============================================================================

def train_and_predict(
    model: NeuralProphet,
    df_train: pd.DataFrame,
    df_full:  pd.DataFrame,
    label:    str,
) -> tuple:
    """
    Fit on df_train, then generate historic predictions across df_full.
    Returns (metrics_df, forecast_df_with_ds_aligned).

    KEY FIX: We merge the returned forecast on `ds` so row positions in the
    forecast dataframe always correspond to the correct calendar dates, even
    when NeuralProphet drops the first n_lags rows internally.
    """
    print(f"\n{'='*60}")
    print(f"  Training : {label}")
    print(f"  Train rows : {len(df_train)}  |  Full rows : {len(df_full)}")
    print(f"{'='*60}")

    metrics = model.fit(df_train, freq="D")

    # Generate predictions for the FULL window (incl. test period)
    future   = model.make_future_dataframe(
        df_full, periods=0, n_historic_predictions=True
    )
    forecast = model.predict(future)

    # ── KEY FIX: align by ds date — keep ALL columns so plot_components()
    #    can access 'trend', 'season_yearly', AR components, etc.
    #    We only use ds as the join key; every forecast column is preserved.
    forecast_aligned = df_full[["ds"]].merge(forecast, on="ds", how="left")

    return metrics, forecast_aligned

## Section 7 — Evaluate

In [32]:
# ==============================================================================
#  SECTION 7 — EVALUATE
# ==============================================================================

def evaluate(y_true: np.ndarray, y_pred: np.ndarray, label: str) -> dict:
    # Drop any NaN pairs (from alignment)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_val = y_true.mean()

    direction_actual = np.sign(np.diff(y_true))
    direction_pred   = np.sign(np.diff(y_pred))
    dir_acc = np.mean(direction_actual == direction_pred) * 100

    results = dict(
        label    = label,
        MAE      = mae,
        RMSE     = rmse,
        MAE_pct  = mae  / mean_val * 100,
        RMSE_pct = rmse / mean_val * 100,
        Dir_Acc  = dir_acc,
    )

    print(f"\n  ── {label} ──────────────────────────")
    print(f"  MAE       : {mae:>10,.2f}   ({results['MAE_pct']:.2f}%)")
    print(f"  RMSE      : {rmse:>10,.2f}   ({results['RMSE_pct']:.2f}%)")
    print(f"  Dir. Acc. : {dir_acc:.2f}%")
    return results

## Section 8 — Visualise Fundamental Features

In [33]:
# ==============================================================================
#  SECTION 8 — VISUALISE FUNDAMENTAL FEATURES
# ==============================================================================

def plot_fundamental_features(fund_q: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle("BBCA Quarterly Fundamental Features (Financial Report JSON)",
                 fontsize=13, fontweight="bold")

    labels = [f"{r.ds.year}-Q{(r.ds.month - 1)//3 + 1}"
              for _, r in fund_q.iterrows()]
    x = range(len(labels))

    axes[0].bar(x, fund_q["net_income"],             color="steelblue")
    axes[0].set_title("Net Income (IDR Billion)")
    axes[1].bar(x, fund_q["revenue"],                color="darkorange")
    axes[1].set_title("Revenue (IDR Billion)")
    axes[2].bar(x, fund_q["net_profit_margin"] * 100, color="seagreen")
    axes[2].set_title("Net Profit Margin (%)")

    for ax in axes:
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels, rotation=45)
        ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig("fundamental_features.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Saved] fundamental_features.png")

## Section 9 — Visualise Model Comparison

In [34]:
# ==============================================================================
#  SECTION 9 — VISUALISE MODEL COMPARISON  (4-panel figure)
# ==============================================================================

def plot_comparison(
    df_price    : pd.DataFrame,
    forecast_bl : pd.DataFrame,
    forecast_hy : pd.DataFrame,
    metrics_bl  : dict,
    metrics_hy  : dict,
    test_size   : int,
) -> None:
    """
    KEY FIX: extract test rows by date comparison (not fragile iloc positions).
    Both forecast dataframes are already date-aligned from train_and_predict().
    """

    # ── Extract only the columns needed for plotting (yhat1 + quantiles) ────────
    # Both forecast dfs may have many component columns — we pull only what we
    # need here so renames don't collide with the full df passed to plot_components
    q_lo_col = next((c for c in forecast_hy.columns
                     if "yhat" in c and "5"  in c and c != "yhat1"), None)
    q_hi_col = next((c for c in forecast_hy.columns
                     if "yhat" in c and "95" in c and c != "yhat1"), None)

    bl_plot_cols = ["ds", "yhat1"]
    hy_plot_cols = ["ds", "yhat1"] + ([q_lo_col] if q_lo_col else []) \
                                   + ([q_hi_col] if q_hi_col else [])

    merged_bl = df_price.merge(
        forecast_bl[bl_plot_cols].rename(columns={"yhat1": "yhat1_bl"}),
        on="ds", how="left"
    )
    merged_hy = df_price.merge(
        forecast_hy[hy_plot_cols].rename(columns={"yhat1": "yhat1_hy"}),
        on="ds", how="left"
    )

    # ── Date-based test/train split ────────────────────────────────────────────
    cutoff_date  = df_price["ds"].iloc[-test_size]
    train_mask   = df_price["ds"] < cutoff_date
    test_mask    = df_price["ds"] >= cutoff_date

    train_actual = df_price[train_mask]
    test_actual  = df_price[test_mask]
    test_bl      = merged_bl[test_mask]
    test_hy      = merged_hy[test_mask]

    # ── Figure ─────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 14))
    fig.suptitle("BBCA NeuralProphet — Baseline vs Hybrid Multimodal Model",
                 fontsize=15, fontweight="bold", y=0.99)
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

    # Panel (a) — full history + test
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(train_actual["ds"], train_actual["y"],
             color="steelblue", linewidth=0.8, label="Training Data")
    ax1.plot(test_actual["ds"],  test_actual["y"],
             color="green",      linewidth=1.2, label="Actual Price (Test)")
    ax1.plot(test_bl["ds"],      test_bl["yhat1_bl"],
             "b--", linewidth=1.1, label="Baseline Prediction")
    ax1.plot(test_hy["ds"],      test_hy["yhat1_hy"],
             "r--", linewidth=1.1, label="Hybrid Prediction")
    ax1.set_title("Full History + Test Period")
    ax1.set_xlabel("Date"); ax1.set_ylabel("Close Price (IDR)")
    ax1.legend(loc="upper left"); ax1.grid(True, alpha=0.3)

    # Panel (b) — zoomed test window
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(test_actual["ds"], test_actual["y"],
             "g.-", linewidth=1.2, label="Actual Price")
    ax2.plot(test_bl["ds"], test_bl["yhat1_bl"],
             "b--", linewidth=1.1,
             label=f"Baseline  MAE={metrics_bl['MAE']:,.0f} ({metrics_bl['MAE_pct']:.1f}%)")
    ax2.plot(test_hy["ds"], test_hy["yhat1_hy"],
             "r--", linewidth=1.1,
             label=f"Hybrid    MAE={metrics_hy['MAE']:,.0f} ({metrics_hy['MAE_pct']:.1f}%)")
    if q_lo_col and q_hi_col and q_lo_col in test_hy.columns:
        ax2.fill_between(test_hy["ds"],
                         test_hy[q_lo_col], test_hy[q_hi_col],
                         alpha=0.15, color="red", label="Hybrid 90% CI")
    ax2.set_title(f"Zoomed View — Last {test_size} Trading Days (Test Set)")
    ax2.set_xlabel("Date"); ax2.set_ylabel("Close Price (IDR)")
    ax2.legend(loc="upper left"); ax2.grid(True, alpha=0.3)

    # Panel (c) — error metrics bar chart
    ax3 = fig.add_subplot(gs[2, 0])
    metric_names = ["MAE (%)", "RMSE (%)"]
    bl_vals = [metrics_bl["MAE_pct"], metrics_bl["RMSE_pct"]]
    hy_vals = [metrics_hy["MAE_pct"], metrics_hy["RMSE_pct"]]
    x = np.arange(len(metric_names))
    width = 0.35
    b1 = ax3.bar(x - width/2, bl_vals, width, label="Baseline",
                 color="steelblue", alpha=0.85)
    b2 = ax3.bar(x + width/2, hy_vals, width, label="Hybrid",
                 color="tomato",    alpha=0.85)
    ax3.bar_label(b1, fmt="%.2f%%", padding=2, fontsize=8)
    ax3.bar_label(b2, fmt="%.2f%%", padding=2, fontsize=8)
    ax3.set_title("Error Metrics (lower is better)")
    ax3.set_xticks(x); ax3.set_xticklabels(metric_names)
    ax3.set_ylabel("% of mean price"); ax3.legend()
    ax3.grid(True, alpha=0.3, axis="y")

    # Panel (d) — directional accuracy
    ax4 = fig.add_subplot(gs[2, 1])
    models   = ["Baseline", "Hybrid"]
    dir_accs = [metrics_bl["Dir_Acc"], metrics_hy["Dir_Acc"]]
    bars = ax4.bar(models, dir_accs,
                   color=["steelblue", "tomato"], alpha=0.85, width=0.4)
    ax4.bar_label(bars, fmt="%.1f%%", padding=2, fontsize=9)
    ax4.axhline(50, linestyle="--", color="grey", linewidth=0.8,
                label="Random baseline (50%)")
    ax4.set_title("Directional Accuracy (higher is better)")
    ax4.set_ylabel("Accuracy (%)"); ax4.set_ylim(0, 100)
    ax4.legend(); ax4.grid(True, alpha=0.3, axis="y")

    plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Saved] model_comparison.png")

## Section 10 — Interpretability: Component Decomposition

In [35]:
# ==============================================================================
#  SECTION 10 — INTERPRETABILITY: MANUAL COMPONENT DECOMPOSITION PLOT
#
#  We do NOT call model.plot_components() because it internally tries to use
#  plotly-resampler (even after set_plotting_backend("matplotlib")) and hangs
#  indefinitely when that package is not installed.
#
#  Instead we read the decomposition columns that NeuralProphet writes into the
#  forecast dataframe directly and build our own matplotlib figure.  This is
#  faster, portable, and gives exactly the same information.
#
#  Column names written by NeuralProphet (version-dependent prefixes):
#    trend                   — long-term trend component
#    season_yearly           — yearly seasonality
#    season_weekly           — weekly seasonality
#    season_quarterly        — custom quarterly seasonality (if added)
#    ar1 … ar<n_lags>        — autoregressive lag contributions (summed)
#    lagged_regressor_<name> — contribution of each external regressor
# ==============================================================================

def plot_components_and_regressors(
    forecast_hy: pd.DataFrame,
) -> None:
    print("\n[Interpretability] Plotting hybrid model component decomposition ...")

    fc = forecast_hy.dropna(subset=["yhat1"]).copy()

    # ── Discover which component columns are present ───────────────────────────
    def _find(keywords: list) -> str | None:
        """Return first column whose name contains ALL keywords (case-insensitive)."""
        for col in fc.columns:
            cl = col.lower()
            if all(k in cl for k in keywords):
                return col
        return None

    trend_col    = _find(["trend"])
    yearly_col   = _find(["season", "year"])
    weekly_col   = _find(["season", "week"])
    quarterly_col= _find(["season", "quart"])

    # Sum all AR lag columns into one composite signal
    ar_cols = [c for c in fc.columns if c.lower().startswith("ar")]
    ar_sum  = fc[ar_cols].sum(axis=1) if ar_cols else None

    # Regressor contribution columns
    reg_cols = [c for c in fc.columns
                if "lagged_regressor" in c.lower() or
                any(r in c.lower() for r in REGRESSOR_COLS)]
    # deduplicate: keep only columns that exist in fc
    reg_cols = [c for c in reg_cols if c in fc.columns]

    # ── Build subplot layout dynamically ──────────────────────────────────────
    component_specs = []
    if trend_col:    component_specs.append(("Trend",            fc[trend_col]))
    if yearly_col:   component_specs.append(("Yearly Seasonality", fc[yearly_col]))
    if weekly_col:   component_specs.append(("Weekly Seasonality", fc[weekly_col]))
    if quarterly_col:component_specs.append(("Quarterly Seasonality", fc[quarterly_col]))
    if ar_sum is not None and ar_cols:
        component_specs.append(("AR Component (sum of lags)", ar_sum))

    # Add each regressor on its own panel
    for rc in reg_cols:
        label = rc.replace("lagged_regressor_", "").replace("_", " ").title()
        component_specs.append((f"Regressor: {label}", fc[rc]))

    if not component_specs:
        print("  [Warning] No decomposition columns found in forecast — "
              "skipping component plot.\n"
              f"  Available columns: {list(fc.columns)}")
        return

    n_panels = len(component_specs)
    fig, axes = plt.subplots(n_panels, 1, figsize=(14, 3 * n_panels), sharex=True)
    if n_panels == 1:
        axes = [axes]

    fig.suptitle(
        "Hybrid Model Component Decomposition\n"
        "(Trend | Seasonality | Autoregression | Fundamental Regressors)",
        fontsize=12, fontweight="bold"
    )

    for ax, (title, series) in zip(axes, component_specs):
        ax.plot(fc["ds"], series, linewidth=0.9, color="steelblue")
        ax.axhline(0, color="grey", linewidth=0.6, linestyle="--")
        ax.set_title(title, fontsize=10)
        ax.set_ylabel("Contribution")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Date")
    plt.tight_layout()
    plt.savefig("hybrid_components.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Saved] hybrid_components.png")

## Section 11 — Summary Table

In [36]:
# ==============================================================================
#  SECTION 11 — SUMMARY TABLE
# ==============================================================================

def print_summary(metrics_bl: dict, metrics_hy: dict) -> None:
    improvement_mae  = (metrics_bl["MAE"]  - metrics_hy["MAE"])  / metrics_bl["MAE"]  * 100
    improvement_rmse = (metrics_bl["RMSE"] - metrics_hy["RMSE"]) / metrics_bl["RMSE"] * 100

    print("\n" + "=" * 62)
    print("  FINAL RESULTS SUMMARY")
    print("=" * 62)
    print(f"  {'Metric':<30} {'Baseline':>12} {'Hybrid':>12}")
    print(f"  {'-'*54}")
    print(f"  {'MAE (IDR)':<30} {metrics_bl['MAE']:>12,.2f} {metrics_hy['MAE']:>12,.2f}")
    print(f"  {'MAE (%)':<30} {metrics_bl['MAE_pct']:>11.2f}% {metrics_hy['MAE_pct']:>11.2f}%")
    print(f"  {'RMSE (IDR)':<30} {metrics_bl['RMSE']:>12,.2f} {metrics_hy['RMSE']:>12,.2f}")
    print(f"  {'RMSE (%)':<30} {metrics_bl['RMSE_pct']:>11.2f}% {metrics_hy['RMSE_pct']:>11.2f}%")
    print(f"  {'Directional Accuracy':<30} {metrics_bl['Dir_Acc']:>11.2f}% {metrics_hy['Dir_Acc']:>11.2f}%")
    print(f"  {'-'*54}")
    print(f"  MAE  improvement (Hybrid → Baseline) : {improvement_mae:+.2f}%")
    print(f"  RMSE improvement (Hybrid → Baseline) : {improvement_rmse:+.2f}%")
    print("=" * 62)


# ==============================================================================
#  MAIN
# ==============================================================================

def main():
    # ── 1. Load ────────────────────────────────────────────────────────────────
    df_price = load_price_data(PRICE_DATA_PATH)
    fund_q   = load_fundamental_data(FUNDAMENTAL_PATH)

    # ── 2. Visualise quarterly fundamentals ───────────────────────────────────
    plot_fundamental_features(fund_q)

    # ── 3. Align fundamentals → daily; normalise regressors ───────────────────
    df_hybrid        = align_fundamentals_to_daily(df_price, fund_q)
    df_hybrid        = normalise_regressors(df_hybrid, REGRESSOR_COLS)
    df_price_aligned = df_hybrid[["ds", "y"]].copy()  # same date range for fair comparison

    # ── 4. Train / test split (by position) ───────────────────────────────────
    df_train_bl = df_price_aligned.iloc[:-TEST_SIZE].copy()
    df_train_hy = df_hybrid.iloc[:-TEST_SIZE].copy()

    # ── 5. BASELINE — fit on train, predict full window ────────────────────────
    model_baseline = build_model(with_regressors=False)
    _, forecast_baseline = train_and_predict(
        model_baseline,
        df_train_bl,
        df_price_aligned,
        "Baseline (Price-Only)",
    )

    # ── 6. HYBRID — fit on train, predict full window ──────────────────────────
    model_hybrid = build_model(with_regressors=True)
    _, forecast_hybrid = train_and_predict(
        model_hybrid,
        df_train_hy[["ds", "y"] + REGRESSOR_COLS],
        df_hybrid[["ds", "y"] + REGRESSOR_COLS],
        "Hybrid (Price + Fundamentals)",
    )

    # ── 7. Extract test-period actuals and predictions by DATE ─────────────────
    cutoff_date = df_price_aligned["ds"].iloc[-TEST_SIZE]

    y_true = df_price_aligned.loc[df_price_aligned["ds"] >= cutoff_date, "y"].values

    y_pred_bl = (
        forecast_baseline
        .loc[forecast_baseline["ds"] >= cutoff_date, "yhat1"]
        .values
    )
    y_pred_hy = (
        forecast_hybrid
        .loc[forecast_hybrid["ds"] >= cutoff_date, "yhat1"]
        .values
    )

    # Trim to equal length in case of any off-by-one from NaN dropping
    min_len   = min(len(y_true), len(y_pred_bl), len(y_pred_hy))
    y_true    = y_true[-min_len:]
    y_pred_bl = y_pred_bl[-min_len:]
    y_pred_hy = y_pred_hy[-min_len:]

    # ── 8. Evaluate ────────────────────────────────────────────────────────────
    print("\n" + "=" * 62)
    print(f"  EVALUATION RESULTS  (last {TEST_SIZE} trading days)")
    print("=" * 62)
    metrics_bl = evaluate(y_true, y_pred_bl, "Baseline (Price-Only)")
    metrics_hy = evaluate(y_true, y_pred_hy, "Hybrid (Price + Fundamentals)")

    # ── 9. Comparison plots ───────────────────────────────────────────────────
    plot_comparison(
        df_price_aligned,
        forecast_baseline,
        forecast_hybrid,
        metrics_bl,
        metrics_hy,
        TEST_SIZE,
    )

    # ── 10. Interpretability: component decomposition ─────────────────────────
    plot_components_and_regressors(forecast_hybrid)

    # ── 11. Summary ───────────────────────────────────────────────────────────
    print_summary(metrics_bl, metrics_hy)


if __name__ == "__main__":
    main()

[Price Data]  2017-07-31 → 2022-07-29 (1256 trading days)

[Fundamental Data]  9 quarterly observations (2020-03-31 → 2022-03-31)
        ds  net_income  revenue  net_profit_margin  ni_growth_qoq  rev_growth_qoq
2020-03-31      6581.0  21769.0           0.302311       0.000000        0.000000
2020-06-30      5659.0  20151.0           0.280830      -0.140100       -0.074326
2020-09-30      7795.0  20281.0           0.384350       0.377452        0.006451
2020-12-31      7096.0  16362.0           0.433688      -0.089673       -0.193235
2021-03-31      7040.0  20558.0           0.342446      -0.007892        0.256448
2021-06-30      7416.0  20667.0           0.358833       0.053409        0.005302
2021-09-30      8743.0  20571.0           0.425016       0.178937       -0.004645
2021-12-31      8224.0  18511.0           0.444276      -0.059362       -0.100141
2022-03-31      8064.0  21050.0           0.383088      -0.019455        0.137162
[Saved] fundamental_features.png

[Hybrid Dataset]

## Run All

In [37]:
main()

[Price Data]  2017-07-31 → 2022-07-29 (1256 trading days)

[Fundamental Data]  9 quarterly observations (2020-03-31 → 2022-03-31)
        ds  net_income  revenue  net_profit_margin  ni_growth_qoq  rev_growth_qoq
2020-03-31      6581.0  21769.0           0.302311       0.000000        0.000000
2020-06-30      5659.0  20151.0           0.280830      -0.140100       -0.074326
2020-09-30      7795.0  20281.0           0.384350       0.377452        0.006451
2020-12-31      7096.0  16362.0           0.433688      -0.089673       -0.193235
2021-03-31      7040.0  20558.0           0.342446      -0.007892        0.256448
2021-06-30      7416.0  20667.0           0.358833       0.053409        0.005302
2021-09-30      8743.0  20571.0           0.425016       0.178937       -0.004645
2021-12-31      8224.0  18511.0           0.444276      -0.059362       -0.100141
2022-03-31      8064.0  21050.0           0.383088      -0.019455        0.137162
[Saved] fundamental_features.png

[Hybrid Dataset]